# Prepare Balanced 30k Dataset for Model Retraining

Builds a clean, balanced 30,000-row dataset using **Reddit data only**:
- **Label 1** (depressed): sampled from `data/raw/reddit_depression_dataset.csv`
- **Label 0** (happy/positive): from `data/raw/happiness_reddit.csv` (scraped positive subreddits)

Outputs train/val/test splits (70/15/15) for retraining LogReg, DistilBERT, and MentalRoBERTa.

> **Prerequisite:** Run `scrape_hapiness_data.ipynb` first to populate `happiness_reddit.csv` with ~15k rows.

In [ ]:
import re
import html
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TARGET_PER_CLASS = 15_000  # 15k label=0 + 15k label=1 = 30k total

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def clean_text(text: str) -> str:
    """Normalize Reddit post text for NLP. Matches pipeline in stan_cleaning_emotion.ipynb."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)  # &amp; → &
    text = re.sub(r"https?://\S+", "", text)  # remove URLs
    text = re.sub(r"r/\w+", "", text)  # remove subreddit mentions
    text = re.sub(r"\.{3,}", " three_dots ", text)  # ... → three_dots
    text = re.sub(r"[^\w\s!?']", " ", text)  # remove punctuation except ! ? '
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)  # collapse repeated chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# --- Load label=1 (depressed) from Reddit Depression Dataset ---
print("Loading reddit_depression_dataset.csv ...")
df_dep = pd.read_csv(
    RAW_DIR / "reddit_depression_dataset.csv",
    usecols=["title", "body", "label"],
    low_memory=False,
)

# Keep only depressed posts
df_dep = df_dep[df_dep["label"] == 1].copy()
print(f"  Depressed rows available: {len(df_dep):,}")

# Merge title + body → text
df_dep["text"] = (df_dep["title"].fillna("") + " " + df_dep["body"].fillna("")).str.strip()

# Drop empty / deleted
df_dep = df_dep[~df_dep["text"].isin(["", "[deleted]", "[removed]"])]
df_dep["text"] = df_dep["text"].apply(clean_text)
df_dep = df_dep[df_dep["text"].str.len() > 10]  # remove near-empty after cleaning

# Sample TARGET_PER_CLASS rows
df_dep = df_dep[["text", "label"]].sample(TARGET_PER_CLASS, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"  Sampled: {len(df_dep):,} depressed posts")

In [ ]:
# --- Load label=0 (happy) from scraped happiness_reddit.csv ---
print("Loading happiness_reddit.csv ...")
df_happy = pd.read_csv(RAW_DIR / "happiness_reddit.csv")
print(f"  Rows available: {len(df_happy):,}")

# The `text` column is already merged title+body from the scraping notebook
df_happy = df_happy[["text", "label"]].copy()
df_happy["label"] = 0  # ensure label=0

# Drop empty
df_happy = df_happy[df_happy["text"].notna() & (df_happy["text"].str.len() > 0)]
df_happy["text"] = df_happy["text"].apply(clean_text)
df_happy = df_happy[df_happy["text"].str.len() > 10]

# Deduplicate
df_happy = df_happy.drop_duplicates(subset="text").reset_index(drop=True)

# Cap at TARGET_PER_CLASS
if len(df_happy) >= TARGET_PER_CLASS:
    df_happy = df_happy.sample(TARGET_PER_CLASS, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"  Sampled: {len(df_happy):,} happy posts")
else:
    print(f"  WARNING: only {len(df_happy):,} happy posts available (target {TARGET_PER_CLASS:,}).")
    print("  Re-run scrape_hapiness_data.ipynb with more subreddits to reach the target.")

In [ ]:
# --- Balance classes to equal size ---
n = min(len(df_dep), len(df_happy))
if n < TARGET_PER_CLASS:
    print(f"Balancing to {n:,} rows per class (less than target {TARGET_PER_CLASS:,}).")

df_dep = df_dep.sample(n, random_state=RANDOM_STATE).reset_index(drop=True)
df_happy = df_happy.sample(n, random_state=RANDOM_STATE).reset_index(drop=True)

# Combine and shuffle
df = pd.concat([df_dep, df_happy], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\nFinal dataset shape: {df.shape}")
print(df["label"].value_counts())

In [ ]:
# --- Stats ---
print(f"Total rows:      {len(df):,}")
print(f"Label 0 (happy): {(df['label'] == 0).sum():,}")
print(f"Label 1 (dep):   {(df['label'] == 1).sum():,}")
print(f"Avg text length: {df['text'].str.len().mean():.0f} chars")
print(f"Min text length: {df['text'].str.len().min()} chars")
print(f"Max text length: {df['text'].str.len().max()} chars")
df.head()

In [ ]:
# --- Train / Val / Test split (70 / 15 / 15) ---
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=RANDOM_STATE)

print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
assert len(train_df) + len(val_df) + len(test_df) == len(df)

In [ ]:
# --- Save to CSV ---
df.to_csv(PROCESSED_DIR / "balanced_dataset_30k.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train_30k.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val_30k.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_30k.csv", index=False)

print("Saved:")
for fname in ["balanced_dataset_30k.csv", "train_30k.csv", "val_30k.csv", "test_30k.csv"]:
    path = PROCESSED_DIR / fname
    print(f"  {path.resolve()}  ({path.stat().st_size / 1024:.0f} KB)")